# EV Charging Station Recommender Pipeline
This notebook runs the full backend data processing and ML pipeline sequentially.

In [3]:
import pandas as pd
import joblib
import os

# --- 1. CONFIGURATION & INPUT ---
USER_INPUT = {
    'city': "Atlanta",
    'state': "GA",
    'charger_type': "DC Fast Charge",
    'min_power_kw':50,
    'network': "ChargePoint",
    'day_of_week': 0,
    'hour_of_day': 17
}

def load_pipeline_data(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Dataset not found at {filepath}")
    df = pd.read_csv(filepath)
    
    # Deduplicate: Keep only the most recent entry per station
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by='timestamp', ascending=False)
    df = df.drop_duplicates(subset=['station_id'], keep='first')
    
    cols_to_drop = [
        'timestamp', 'latitude', 'longitude', 'location_type', 'amenities_nearby',
        'ports_available', 'precipitation_mm', 'temperature_f', 'weather_condition',
        'gas_price_per_gallon', 'local_event', 'is_weekend', 'month'
    ]
    return df.drop(columns=cols_to_drop)

def ensure_models():
    model_paths = [
        'models/wait_time_model.pkl', 
        'models/high_utilization_model.pkl', 
        'models/duration_model.pkl',
        'models/price_model.pkl'
    ]
    return all(os.path.exists(p) for p in model_paths)

if not ensure_models():
    print("CRITICAL: Models not found. Please run models/train_models.ipynb first.")
else:
    df = load_pipeline_data('data/ev_stations.csv')
    wait_model = joblib.load('models/wait_time_model.pkl')
    peak_model = joblib.load('models/high_utilization_model.pkl')
    duration_model = joblib.load('models/duration_model.pkl')
    price_model = joblib.load('models/price_model.pkl')
    
    print("Starting Sequential Pipeline...")
    
    # --- TASK EXECUTION SEQUENCE ---
    
    # Task 1: Filter by Region
    %run tasks/task1_filter.py
    df = filter_by_region(df, USER_INPUT['city'], USER_INPUT['state'])
    print(f"Task 1 Complete: {len(df)} stations remain.")
    
    if not df.empty:
        # Task 2: Hardware Compatibility
        %run tasks/task2_hardware.py
        df = hardware_compatibility(df, USER_INPUT['charger_type'], USER_INPUT['min_power_kw'], USER_INPUT['network'])
        print(f"Task 2 Complete: {len(df)} stations remain.")
        
        if not df.empty:
            # Task 3: Wait Time (Model A Refactored to Classifier)
            %run tasks/task3_wait_time.py
            df = predict_wait_time(df, wait_model)
            print("Task 3 Complete: Queue probabilities predicted.")
            
            # Task 4: High Utilization (Model B Refactored)
            %run tasks/task4_high_utilization.py
            df = predict_high_utilization(df, peak_model, USER_INPUT['day_of_week'], USER_INPUT['hour_of_day'])
            print("Task 4 Complete: High utilization probabilities added.")
            
            # Task 5: Reliability
            %run tasks/task5_reliability.py
            df = compute_reliability(df)
            print("Task 5 Complete: Reliability scores computed.")
            
            # Task 6: Duration
            %run tasks/task6_duration.py
            df = predict_duration(df, duration_model)
            print("Task 6 Complete: Predicted durations added.")
            
            # Task 7: Cost (Includes Price Prediction from Model D)
            %run tasks/task7_cost.py
            df = estimate_cost(df, price_model)
            print("Task 7 Complete: Dynamic prices and total costs estimated.")
            
            # Task 8: Ranking
            %run tasks/task8_ranking.py
            top_5 = hybrid_ranking(df)
            print("Task 8 Complete: Top 5 stations ranked.")
            
            # --- FINAL OUTPUT ---
            print("\n--- TOP 5 RECOMMENDED STATIONS ---")
            print(top_5[['station_id', 'station_name', 'city', 'state', 'charger_type', 'power_output_kw', 'final_score']].to_string(index=False))
        else:
            print("No compatible stations found.")
    else:
        print("No stations found in this region.")


Starting Sequential Pipeline...
Task 1 Complete: 11 stations remain.
Task 2 Complete: 7 stations remain.
Task 3 Complete: Queue probabilities predicted.
Task 4 Complete: High utilization probabilities added.
Task 5 Complete: Reliability scores computed.
Task 6 Complete: Predicted durations added.
Task 7 Complete: Dynamic prices and total costs estimated.
Task 8 Complete: Top 5 stations ranked.

--- TOP 5 RECOMMENDED STATIONS ---
station_id                 station_name    city state   charger_type  power_output_kw  final_score
   EV00099           EVgo - Atlanta #19 Atlanta    GA DC Fast Charge            350.0     0.929368
   EV00029     ChargePoint - Atlanta #9 Atlanta    GA DC Fast Charge             62.5     0.883769
   EV00088  Shell Recharge - Atlanta #8 Atlanta    GA DC Fast Charge            350.0     0.758823
   EV00036 Shell Recharge - Atlanta #16 Atlanta    GA DC Fast Charge            350.0     0.738690
   EV00150           EVCS - Atlanta #10 Atlanta    GA DC Fast Charge    